In [7]:
import pandas as pd

trades = pd.read_excel(  r"C:\Keshav S\Investements Operations Platform\Trades.xlsx")
broker = pd.read_excel(
    r"C:\Keshav S\Investements Operations Platform\Broker.xlsx"
)
print (trades.shape)
print(broker.shape)
trades.head()
broker.head()

(500, 9)
(495, 9)


,Trade_ID,Trade_Date,Security,Asset_Class,Buy_Sell,Quantity,Trade_Price,Broker,Trader
0,T0001,2025-07-01,GOOG,Equity,Sell,173,559.19,Bank of America,Trader A
1,T0002,2025-07-01,AAPL,Equity,Buy,121,707.05,Goldman Sachs,Trader A
2,T0003,2025-07-01,MSFT,Equity,Sell,83,201.86,Bank of America,Trader C
3,T0004,2025-07-01,BAC,Equity,Buy,130,336.83,Bank of America,Trader A
4,T0005,2025-07-01,AAPL,Equity,Buy,226,366.29,Goldman Sachs,Trader B


In [8]:
print(trades.columns)

Index(['Trade_ID', 'Trade_Date', 'Security', 'Asset_Class', 'Buy_Sell',
       'Quantity', 'Trade_Price', 'Broker', 'Trader'],
      dtype='object')


In [11]:
#merge trades and broker
recon = pd.merge(trades, broker,on= "Trade_ID", how = "left", suffixes =("_Internal","_Broker"))
print(recon.columns)

Index(['Trade_ID', 'Trade_Date_Internal', 'Security_Internal',
       'Asset_Class_Internal', 'Buy_Sell_Internal', 'Quantity_Internal',
       'Trade_Price_Internal', 'Broker_Internal', 'Trader_Internal',
       'Trade_Date_Broker', 'Security_Broker', 'Asset_Class_Broker',
       'Buy_Sell_Broker', 'Quantity_Broker', 'Trade_Price_Broker',
       'Broker_Broker', 'Trader_Broker'],
      dtype='object')


In [10]:
broker.head()

,Trade_ID,Trade_Date,Security,Asset_Class,Buy_Sell,Quantity,Trade_Price,Broker,Trader
0,T0001,2025-07-01,GOOG,Equity,Sell,173,559.19,Bank of America,Trader A
1,T0002,2025-07-01,AAPL,Equity,Buy,121,707.05,Goldman Sachs,Trader A
2,T0003,2025-07-01,MSFT,Equity,Sell,83,201.86,Bank of America,Trader C
3,T0004,2025-07-01,BAC,Equity,Buy,130,336.83,Bank of America,Trader A
4,T0005,2025-07-01,AAPL,Equity,Buy,226,366.29,Goldman Sachs,Trader B


In [12]:
import pandas as pd
import numpy as np

# Load files
trades = pd.read_excel(
    r"C:\Keshav S\Investements Operations Platform\Trades.xlsx"
)

broker = pd.read_excel(
    r"C:\Keshav S\Investements Operations Platform\Broker.xlsx"
)

# Merge internal and broker records
recon = pd.merge(
    trades,
    broker,
    on="Trade_ID",
    how="left",
    suffixes=("_Internal", "_Broker")
)

# Create Exception Type column
recon["Exception_Type"] = np.nan

# Missing Trade
recon.loc[
    recon["Quantity_Broker"].isna(),
    "Exception_Type"
] = "Missing in Broker"

# Quantity Mismatch
recon.loc[
    (recon["Quantity_Broker"].notna()) &
    (recon["Quantity_Internal"] != recon["Quantity_Broker"]),
    "Exception_Type"
] = "Quantity Mismatch"

# Price Mismatch
recon.loc[
    (recon["Quantity_Broker"].notna()) &
    (recon["Trade_Price_Internal"] != recon["Trade_Price_Broker"]),
    "Exception_Type"
] = "Price Mismatch"

# Keep only exception rows
exceptions = recon[
    recon["Exception_Type"].notna()
]

# Select useful columns
exception_report = exceptions[
    [
        "Trade_ID",
        "Security_Internal",
        "Quantity_Internal",
        "Quantity_Broker",
        "Trade_Price_Internal",
        "Trade_Price_Broker",
        "Exception_Type"
    ]
]

# Save report
exception_report.to_excel(
    r"C:\Keshav S\Investements Operations Platform\Exception_Report.xlsx",
    index=False
)

# Summary
print("Total Trades:", len(trades))
print("Exceptions Found:", len(exception_report))
print("\nBreakdown:")
print(exception_report["Exception_Type"].value_counts())

Total Trades: 500
Exceptions Found: 24

Breakdown:
Exception_Type
Quantity Mismatch    10
Price Mismatch        9
Missing in Broker     5
Name: count, dtype: int64


C:\Users\Hitish Mehan\AppData\Local\Temp\ipykernel_9932\2619574878.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Missing in Broker' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  recon.loc[
